In [1]:
import os
import sys
import time
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional

import jax
import jax.numpy as jnp
import numpy as np
import optax
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from flax import linen as nn
from tqdm import tqdm

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from parameters.shared_models import (
    MiniGPT,
    custom_sgd,
    custom_sgd_log,
    custom_sgd_rms
)

# Configure plotting aesthetics and Optuna logging
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Ensure deterministic 64-bit precision for numerical stability
jax.config.update("jax_enable_x64", True)

/Users/noah-everett/Documents/Research/Induced-Metric-Optimiser/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration Overview

This section defines global switches controlling experiment depth, Optuna search budgets, and MiniGPT architecture settings.
Use demo mode (default) for quick iteration, then flip `RUN_FULL_EXPERIMENT` to reproduce the full sweep locally.

In [2]:
# Global experiment switches
RUN_FULL_EXPERIMENT = False  # Toggle to True for full 250-epoch sweeps
SEED = 123
np.random.seed(SEED)

# Search/training budgets
N_OPTUNA_TRIALS = 40 if RUN_FULL_EXPERIMENT else 6
N_EPOCHS = 250 if RUN_FULL_EXPERIMENT else 8
VAL_FREQ = 5 if RUN_FULL_EXPERIMENT else 2
BATCH_SIZE = 256 if RUN_FULL_EXPERIMENT else 96
MAX_TRAIN_BATCHES = None if RUN_FULL_EXPERIMENT else 8  # Cap batches for demo speed
DATA_SEQ_LEN = 256 if RUN_FULL_EXPERIMENT else 128
VAL_SPLIT = 0.1
DEVICE = jax.default_backend()

OPTIMISERS = [
    "adam",
    "adamw",
    "sgd",
    "sgd_metric",
    "sgd_log_metric",
    "sgd_rms",
    "muon",
]

ARCHITECTURE_CONFIG = {
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "seq_len": DATA_SEQ_LEN,
    "embed_dim": 128,
    "num_heads": 4,
    "num_layers": 4,
    "dropout_rate": 0.1,
}

print(f"Backend: {DEVICE}")
print(f"Demo mode: {not RUN_FULL_EXPERIMENT}")
print(f"Optuna trials/optimizer: {N_OPTUNA_TRIALS}")
print(f"Epochs per trial: {N_EPOCHS}")

Backend: cpu
Demo mode: True
Optuna trials/optimizer: 6
Epochs per trial: 8


## Tiny Shakespeare Data Pipeline

We reuse the helpers from `sweep_shake.py`, but expose them as reusable notebook functions.

In [3]:
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_DIR = Path.cwd() / ".." / "data"
DATA_FILE = DATA_DIR / "tinyshakespeare.txt"
DATA_SEED = 2024

def download_shakespeare() -> Path:
    """Ensure the Tiny Shakespeare corpus is available locally."""
    if not DATA_FILE.exists():
        resp = requests.get(SHAKESPEARE_URL, timeout=60)
        resp.raise_for_status()
        DATA_FILE.write_text(resp.text, encoding="utf-8")
        print("Downloaded tinyshakespeare.txt")
    return DATA_FILE

@lru_cache(maxsize=1)
def read_corpus() -> str:
    path = download_shakespeare()
    return path.read_text(encoding="utf-8")

@lru_cache(maxsize=None)
def load_shakespeare(seq_len: int, val_split: float = 0.1):
    """Tokenise the corpus and split into train/validation arrays."""
    text = read_corpus()
    chars = sorted(set(text))
    vocab_size = len(chars)
    char_to_idx = {c: i for i, c in enumerate(chars)}
    idx_to_char = {i: c for i, c in enumerate(chars)}

    data = jnp.array([char_to_idx[ch] for ch in text], dtype=jnp.int32)
    split_idx = int(len(data) * (1 - val_split))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    
    info = {
        "train_data": train_data,
        "val_data": val_data,
        "vocab_size": vocab_size,
        "char_to_idx": char_to_idx,
        "idx_to_char": idx_to_char,
        "seq_len": seq_len,
    }
    return info

def create_text_batches(data, seq_len, batch_size, seed, max_batches=None):
    """Construct sequential (input, target) pairs for language modelling."""
    key = jax.random.PRNGKey(seed)
    num_sequences = (len(data) - 1) // seq_len
    inputs = []
    targets = []
    for i in range(num_sequences):
        start = i * seq_len
        end = start + seq_len
        if end >= len(data):
            break
        inputs.append(data[start:end])
        targets.append(data[start + 1:end + 1])
    inputs = jnp.array(inputs)
    targets = jnp.array(targets)

    perm = jax.random.permutation(key, len(inputs))
    inputs = inputs[perm]
    targets = targets[perm]

    num_batches = len(inputs) // batch_size
    batches = []
    for i in range(num_batches):
        s = i * batch_size
        e = s + batch_size
        batches.append((inputs[s:e], targets[s:e]))
        if max_batches is not None and len(batches) >= max_batches:
            break
    return batches

@lru_cache(maxsize=None)
def prepare_batches(seq_len: int, batch_size: int, max_batches: Optional[int]):
    """Cache batches so Optuna trials reuse the same data for fairness."""
    data_info = load_shakespeare(seq_len, VAL_SPLIT)
    train_batches = create_text_batches(
        data_info["train_data"], seq_len, batch_size, DATA_SEED, max_batches
    )
    val_batches = create_text_batches(
        data_info["val_data"], seq_len, batch_size, DATA_SEED + 1, max_batches
    )
    return data_info, train_batches, val_batches

## Training Utilities

Below we define the MiniGPT training loop, optimizer factories, and the Optuna-friendly wrappers.

In [4]:
LOG_METRIC_OPTIMISERS = {"sgd_log_metric"}

@dataclass
class OptimisationResult:
    optimiser: str
    trial_index: int
    hyperparameters: Dict[str, float]
    min_val_perplexity: float
    min_perp_epoch: int
    final_train_loss: float
    final_val_perplexity: float
    train_losses: List[float]
    val_perplexities: List[float]
    runtime_seconds: float
    seed: int

def loss_fn(params, x, y, model, key):
    logits = model.apply(params, x, train=True, rngs={"dropout": key})
    logits_flat = logits.reshape(-1, logits.shape[-1])
    targets_flat = y.reshape(-1)
    return optax.softmax_cross_entropy_with_integer_labels(logits_flat, targets_flat).mean()

def perplexity_fn(params, batches, model):
    if not batches:
        return jnp.inf
    total_loss = 0.0
    total_tokens = 0
    for x_batch, y_batch in batches:
        logits = model.apply(params, x_batch, train=False)
        logits_flat = logits.reshape(-1, logits.shape[-1])
        targets_flat = y_batch.reshape(-1)
        batch_loss = optax.softmax_cross_entropy_with_integer_labels(logits_flat, targets_flat).mean()
        total_loss += batch_loss * y_batch.size
        total_tokens += y_batch.size
    avg_loss = total_loss / total_tokens
    return float(jnp.exp(avg_loss))

def build_optimizer(name: str, hp: Dict[str, float]):
    if name == "adam":
        return optax.adam(
            learning_rate=hp["learning_rate"],
            b1=hp["beta1"],
            b2=hp["beta2"],
            eps=hp["eps"],
        )
    if name == "adamw":
        return optax.adamw(
            learning_rate=hp["learning_rate"],
            b1=hp["beta1"],
            b2=hp["beta2"],
            eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    if name == "sgd":
        return optax.sgd(
            learning_rate=hp["learning_rate"],
            momentum=hp["momentum"],
        )
    if name == "sgd_metric":
        return custom_sgd(
            learning_rate=hp["learning_rate"],
            momentum=hp["momentum"],
            xi=hp["xi"],
            beta=hp["beta"],
            weight_decay=hp["weight_decay"],
        )
    if name == "sgd_log_metric":
        return custom_sgd_log(
            learning_rate=hp["learning_rate"],
            momentum=hp["momentum"],
            xi=hp["xi"],
            beta=hp["beta"],
            weight_decay=hp["weight_decay"],
        )
    if name == "sgd_rms":
        return custom_sgd_rms(
            learning_rate=hp["learning_rate"],
            momentum=hp["momentum"],
            xi=hp["xi"],
            beta=hp["beta"],
            beta_rms=hp["beta_rms"],
            eps=hp["eps"],
            weight_decay=hp["weight_decay"],
        )
    if name == "muon":
        return optax.contrib.muon(
            learning_rate=hp["learning_rate"],
            adam_b1=hp["adam_b1"],
            adam_b2=hp["adam_b2"],
            eps=hp["eps"],
            beta=hp["beta"],
            weight_decay=hp["weight_decay"],
        )
    raise ValueError(f"Unknown optimiser: {name}")

def train_minigpt(optimiser_name: str, hyperparameters: Dict[str, float], seed: int, trial_index: int) -> OptimisationResult:
    data_info, train_batches, val_batches = prepare_batches(
        ARCHITECTURE_CONFIG["seq_len"], ARCHITECTURE_CONFIG["batch_size"], MAX_TRAIN_BATCHES
    )
    seq_len = ARCHITECTURE_CONFIG["seq_len"]
    model = MiniGPT(
        vocab_size=data_info["vocab_size"],
        embed_dim=ARCHITECTURE_CONFIG["embed_dim"],
        num_heads=ARCHITECTURE_CONFIG["num_heads"],
        num_layers=ARCHITECTURE_CONFIG["num_layers"],
        dropout_rate=ARCHITECTURE_CONFIG["dropout_rate"],
        max_seq_len=seq_len,
    )

    base_key = jax.random.PRNGKey(seed)
    init_key, dropout_key = jax.random.split(base_key)
    dummy_input = jnp.ones((1, seq_len), dtype=jnp.int32)
    params = model.init({"params": init_key, "dropout": dropout_key}, dummy_input, train=True)

    optimiser = build_optimizer(optimiser_name, hyperparameters)
    opt_state = optimiser.init(params)

    @jax.jit
    def update_step(params, opt_state, x, y, key):
        def loss_with_dropout(p):
            return loss_fn(p, x, y, model, key)
        loss, grads = jax.value_and_grad(loss_with_dropout)(params)
        if optimiser_name in LOG_METRIC_OPTIMISERS:
            updates, new_state = optimiser.update(grads, opt_state, loss, params)
        else:
            updates, new_state = optimiser.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_state, loss

    train_losses = []
    val_perplexities = []
    min_val_perplexity = float("inf")
    min_perp_epoch = -1

    start_time = time.time()
    epoch_key = jax.random.PRNGKey(seed + 1)
    epoch_keys = jax.random.split(epoch_key, ARCHITECTURE_CONFIG["n_epochs"])
    for epoch in range(ARCHITECTURE_CONFIG["n_epochs"]):
        batch_losses = []
        if not train_batches:
            break
        for batch_idx, (x_batch, y_batch) in enumerate(train_batches):
            batch_key = jax.random.fold_in(epoch_keys[epoch], batch_idx)
            params, opt_state, batch_loss = update_step(params, opt_state, x_batch, y_batch, batch_key)
            batch_losses.append(float(batch_loss))
        epoch_loss = float(np.mean(batch_losses)) if batch_losses else float("nan")
        train_losses.append(epoch_loss)

        if (epoch + 1) % VAL_FREQ == 0 or epoch == ARCHITECTURE_CONFIG["n_epochs"] - 1:
            val_perp = perplexity_fn(params, val_batches, model)
            val_perplexities.append(val_perp)
            if val_perp < min_val_perplexity:
                min_val_perplexity = val_perp
                min_perp_epoch = epoch

    runtime = time.time() - start_time
    final_val = val_perplexities[-1] if val_perplexities else float("inf")

    return OptimisationResult(
        optimiser=optimiser_name,
        trial_index=trial_index,
        hyperparameters=hyperparameters,
        min_val_perplexity=min_val_perplexity,
        min_perp_epoch=min_perp_epoch,
        final_train_loss=train_losses[-1] if train_losses else float("nan"),
        final_val_perplexity=final_val,
        train_losses=train_losses,
        val_perplexities=val_perplexities,
        runtime_seconds=runtime,
        seed=seed,
    )

## Optuna Search per Optimiser

We now wrap the training loop with Optuna so that each optimiser gets its own study with independent hyperparameter proposals.

In [ ]:
def suggest_hyperparameters(trial: optuna.Trial, optimiser_name: str) -> Dict[str, float]:
    if optimiser_name == "adam":
        return {
            "learning_rate": trial.suggest_float("adam_lr", 1e-5, 1e-1, log=True),
            "beta1": trial.suggest_float("adam_b1", 0.8, 0.99),
            "beta2": trial.suggest_float("adam_b2", 0.9, 0.999),
            "eps": trial.suggest_float("adam_eps", 1e-9, 1e-7, log=True),
        }
    if optimiser_name == "adamw":
        return {
            "learning_rate": trial.suggest_float("adamw_lr", 1e-5, 1e-1, log=True),
            "beta1": trial.suggest_float("adamw_b1", 0.8, 0.99),
            "beta2": trial.suggest_float("adamw_b2", 0.9, 0.999),
            "eps": trial.suggest_float("adamw_eps", 1e-9, 1e-7, log=True),
            "weight_decay": trial.suggest_float("adamw_decay", 1e-6, 5e-2, log=True),
        }
    if optimiser_name == "sgd":
        return {
            "learning_rate": trial.suggest_float("sgd_lr", 1e-4, 1.0, log=True),
            "momentum": trial.suggest_float("sgd_momentum", 0.2, 0.99),
        }
    if optimiser_name in {"sgd_metric", "sgd_log_metric"}:
        return {
            "learning_rate": trial.suggest_float("metric_lr", 1e-4, 1.0, log=True),
            "momentum": trial.suggest_float("metric_momentum", 0.5, 0.99),
            "xi": trial.suggest_float("metric_xi", 1e-8, 1e-3, log=True),
            "beta": trial.suggest_float("metric_beta", 0.6, 0.95),
            "weight_decay": trial.suggest_float("metric_decay", 1e-6, 1e-2, log=True),
        }
    if optimiser_name == "sgd_rms":
        return {
            "learning_rate": trial.suggest_float("sgd_rms_lr", 1e-4, 1.0, log=True),
            "momentum": trial.suggest_float("sgd_rms_momentum", 0.6, 0.99),
            "xi": trial.suggest_float("sgd_rms_xi", 1e-8, 1e-3, log=True),
            "beta": trial.suggest_float("sgd_rms_beta", 0.6, 0.95),
            "beta_rms": trial.suggest_float("sgd_rms_beta_rms", 0.9, 0.999),
            "eps": trial.suggest_float("sgd_rms_eps", 1e-9, 1e-6, log=True),
            "weight_decay": trial.suggest_float("sgd_rms_decay", 1e-6, 1e-2, log=True),
        }
    if optimiser_name == "muon":
        return {
            "learning_rate": trial.suggest_float("muon_lr", 1e-5, 5e-2, log=True),
            "adam_b1": trial.suggest_float("muon_b1", 0.8, 0.99),
            "adam_b2": trial.suggest_float("muon_b2", 0.9, 0.999),
            "eps": trial.suggest_float("muon_eps", 1e-9, 1e-7, log=True),
            "beta": trial.suggest_float("muon_beta", 0.9, 0.999),
            "weight_decay": trial.suggest_float("muon_decay", 1e-6, 1e-2, log=True),
        }
    raise ValueError(f"Unknown optimiser: {optimiser_name}")

all_results: List[OptimisationResult] = []
studies: Dict[str, optuna.Study] = {}

for idx, optimiser_name in enumerate(tqdm(OPTIMISERS, desc="Optimisers")):
    local_sampler = optuna.samplers.TPESampler(
        seed=SEED + idx, multivariate=True, constant_liar=True
    )
    study = optuna.create_study(
        direction="minimize", sampler=local_sampler, study_name=f"{optimiser_name}_shake_local"
    )

    def objective(trial: optuna.Trial, name=optimiser_name):
        hparams = suggest_hyperparameters(trial, name)
        seed = SEED + trial.number + idx * 1_000
        result = train_minigpt(name, hparams, seed, trial.number)
        all_results.append(result)
        trial.set_user_attr("min_val_perplexity", result.min_val_perplexity)
        trial.set_user_attr("runtime_seconds", result.runtime_seconds)
        trial.set_user_attr("hyperparameters", result.hyperparameters)
        return result.min_val_perplexity

    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)
    studies[optimiser_name] = study

print("Completed local sweeps for all optimisers.")

Optimisers:   0%|          | 0/7 [00:00<?, ?it/s]/Users/noah-everett/Documents/Research/Induced-Metric-Optimiser/.conda/lib/python3.11/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/Users/noah-everett/Documents/Research/Induced-Metric-Optimiser/.conda/lib/python3.11/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/Users/noah-everett/Documents/Research/Induced-Metric-Optimiser/.conda/lib/python3.11/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/Users/noah-everett/Documents/Research/Induced-Metric-Optimiser/.conda/lib/python3.11/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``constant_l

Downloaded tinyshakespeare.txt


## Summaries and Tables

Transform Optuna runs into tidy DataFrames so we can inspect best hyperparameters per optimiser.

In [ ]:
def results_to_dataframe(results: List[OptimisationResult]) -> pd.DataFrame:
    rows = []
    for res in results:
        rows.append({
            "optimiser": res.optimiser,
            "trial_index": res.trial_index,
            "min_val_perplexity": res.min_val_perplexity,
            "min_perp_epoch": res.min_perp_epoch,
            "final_train_loss": res.final_train_loss,
            "final_val_perplexity": res.final_val_perplexity,
            "runtime_seconds": res.runtime_seconds,
            "hyperparameters": res.hyperparameters,
            "train_curve": res.train_losses,
            "val_curve": res.val_perplexities,
            "seed": res.seed,
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame()

results_df = results_to_dataframe(all_results)
if results_df.empty:
    print("No results collected yet. Run the Optuna cell above first.")
else:
    best_idx = results_df.groupby("optimiser")["min_val_perplexity"].idxmin()
    best_df = results_df.loc[best_idx].sort_values("min_val_perplexity").reset_index(drop=True)
    display(best_df[["optimiser", "min_val_perplexity", "min_perp_epoch", "runtime_seconds", "hyperparameters"]])

    print("\nOverall stats:")
    print(results_df.groupby("optimiser")["min_val_perplexity"].describe()[["mean", "min", "std"]])

## Visualisations

Plot validation perplexity, runtime, and convergence curves for the best run of each optimiser.

In [ ]:
if results_df.empty:
    print("No plots to show yet. Run the optimisation first.")
else:
    ordered_best = best_df.sort_values("min_val_perplexity")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    sns.barplot(
        data=ordered_best,
        x="optimiser",
        y="min_val_perplexity",
        ax=axes[0, 0],
        order=ordered_best["optimiser"],
        palette="viridis")
    axes[0, 0].set_title("Best Validation Perplexity (↓)")
    axes[0, 0].set_ylabel("Perplexity")
    axes[0, 0].tick_params(axis="x", rotation=45)

    sns.barplot(
        data=ordered_best,
        x="optimiser",
        y="runtime_seconds",
        ax=axes[0, 1],
        order=ordered_best["optimiser"],
        palette="magma")
    axes[0, 1].set_title("Runtime of Best Trial (s)")
    axes[0, 1].set_ylabel("Seconds")
    axes[0, 1].tick_params(axis="x", rotation=45)

    axes[1, 0].set_title("Validation Perplexity Curves")
    for _, row in ordered_best.iterrows():
        curve = row["val_curve"]
        if not curve:
            continue
        axes[1, 0].plot(range(len(curve)), curve, label=row["optimiser"])
    axes[1, 0].set_xlabel("Evaluation step")
    axes[1, 0].set_ylabel("Perplexity")
    axes[1, 0].legend()

    axes[1, 1].set_title("Train Loss Curves")
    for _, row in ordered_best.iterrows():
        curve = row["train_curve"]
        if not curve:
            continue
        axes[1, 1].plot(range(len(curve)), curve, label=row["optimiser"])
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

## Best Hyperparameters per Optimiser

In [ ]:
if results_df.empty or "best_df" not in globals():
    print("No hyperparameters to display yet.")
else:
    display_df = best_df.sort_values("min_val_perplexity")
    for _, row in display_df.iterrows():
        print(
            f"{row['optimiser'].upper()} | Perplexity: {row['min_val_perplexity']:.3f} "
            f"| Runtime: {row['runtime_seconds']:.1f}s"
        )
        for key, value in row["hyperparameters"].items():
            print(f"  - {key}: {value}")
        print()